In [1]:
import __init__ as init

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

In [2]:
def apply_column_consolidation(df, mapping):
    """Applies the exact same column renaming to the raw 2011 data that preprocess.py applies to the cleaned data"""
    df = df.rename(columns=mapping)
    df = df.rename(columns=init.tableau_snake_case_names)
    return df

In [3]:
def pre_pca_concat():
    df_2011 = pd.read_csv(f"{init.root_data_clean_joins_dir}2011_cleaned.csv")
    df_2021 = pd.read_csv(f"{init.root_data_clean_joins_dir}2021.csv")

    df_2021 = df_2021.reindex(columns=df_2011.columns)

    df_2011['year'] = 2011
    df_2021['year'] = 2021

    df_master = pd.concat([df_2011, df_2021], ignore_index=True)

    return df_master

In [4]:
def compute_proportions(df):

    normalization_map = init.normalization_map

    df_props = pd.DataFrame(index=df.index)
    
    for denominator, features in normalization_map.items():
        valid_features = [col for col in features if col in df.columns]

        safe_denominator = df[denominator].replace(0, 1)
        
        for feature in valid_features:
            df_props[feature] = df[feature] / safe_denominator

    df_props = df_props.fillna(0)

    return df_props

In [5]:
def main():
    print("Loading datasets...")
    
    # 1. Load Original 2011 (349 rows) 
    df_2011_orig = pd.read_csv(f"{init.root_data_clean_joins_dir}2011.csv")
    df_2011_orig = apply_column_consolidation(df_2011_orig, init.census_ethnicity_mapping_2011)
    df_2011_orig['year'] = 2011

    # 2. Load Mapped 2011 (318 rows) 
    df_2011_mapped = pd.read_csv(f"{init.root_data_clean_joins_dir}2011_cleaned.csv")
    df_2011_mapped['year'] = 2011

    # 3. Load 2021 (318 rows) 
    df_2021 = pd.read_csv(f"{init.root_data_clean_joins_dir}2021.csv")
    df_2021['year'] = 2021

    # Create the Comparison Set (318 mapped + 318)
    df_comparison = pd.concat([df_2011_mapped, df_2021], ignore_index=True)

    print("Computing proportions...")
    props_2011_orig = compute_proportions(df_2011_orig)
    props_comparison = compute_proportions(df_comparison)

    totals_to_exclude = ['white_total', 'asian_total', 'black_total', 'mixed_total', 'other_total']
    pca_features = [col for col in props_2011_orig.columns if col not in totals_to_exclude]

    # =========================================================
    # WORKBOOK 1: 2011 ANALYSIS ONLY (349 rows, PCA & t-SNE)
    # =========================================================
    print("Processing 2011 Analysis PCA & t-SNE...")
    scaler_2011 = StandardScaler()
    scaled_2011 = scaler_2011.fit_transform(props_2011_orig[pca_features])

    # PCA 
    pca = PCA(n_components=4, random_state=42)
    pca_result = pca.fit_transform(scaled_2011)

    loadings = pd.DataFrame(
        pca.components_.T, columns=['PC1_Weight', 'PC2_Weight', 'PC3_Weight', 'PC4_Weight'], index=pca_features 
    )
    loadings.to_csv(f"{init.root_data_results_dir}pca_loadings.csv")

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_result = tsne.fit_transform(scaled_2011)

    # Compile Final 2011 File
    props_2011_orig = props_2011_orig.add_suffix('_prop')
    df_final_2011 = pd.concat([df_2011_orig, props_2011_orig], axis=1)
    
    df_final_2011['PCA_1'] = pca_result[:, 0]
    df_final_2011['PCA_2'] = pca_result[:, 1]
    df_final_2011['PCA_3'] = pca_result[:, 2]
    df_final_2011['PCA_4'] = pca_result[:, 3]
    df_final_2011['TSNE_X'] = tsne_result[:, 0]
    df_final_2011['TSNE_Y'] = tsne_result[:, 1]

    # =========================================================
    # WORKBOOK 2: 2011 vs 2021 COMPARISON (318 + 318 rows, UMAP)
    # =========================================================
    print("Processing Comparison UMAP...")
    scaler_comp = StandardScaler()
    scaled_comp = scaler_comp.fit_transform(props_comparison[pca_features])

    reducer = umap.UMAP(n_components=2, random_state=42)
    umap_result = reducer.fit_transform(scaled_comp)

    # Compile Final Comparison File
    props_comparison = props_comparison.add_suffix('_prop')
    df_final_comparison = pd.concat([df_comparison, props_comparison], axis=1)

    df_final_comparison['UMAP_X'] = umap_result[:, 0]
    df_final_comparison['UMAP_Y'] = umap_result[:, 1]

    # =========================================================
    # EXPORT
    # =========================================================
    out_2011 = f"{init.root_data_results_dir}2011_analysis.csv"
    out_comparison = f"{init.root_data_results_dir}2011_2021_comparison.csv"
    
    df_final_2011.to_csv(out_2011, index=False)
    df_final_comparison.to_csv(out_comparison, index=False)
    
    print(f"Success! Exported:\n1. {out_2011}\n2. {out_comparison}")

In [6]:
main()

Loading datasets...
Computing proportions...
Processing 2011 Analysis PCA & t-SNE...


C:\Users\ksstu\AppData\Local\Temp\ipykernel_42876\1013980722.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final_2011['PCA_1'] = pca_result[:, 0]
C:\Users\ksstu\AppData\Local\Temp\ipykernel_42876\1013980722.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final_2011['PCA_2'] = pca_result[:, 1]
C:\Users\ksstu\AppData\Local\Temp\ipykernel_42876\1013980722.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

Processing Comparison UMAP...
Success! Exported:
1. E:\uob\va-coursework\data/results/2011_analysis.csv
2. E:\uob\va-coursework\data/results/2011_2021_comparison.csv


C:\Users\ksstu\AppData\Local\Temp\ipykernel_42876\1013980722.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final_comparison['UMAP_X'] = umap_result[:, 0]
C:\Users\ksstu\AppData\Local\Temp\ipykernel_42876\1013980722.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final_comparison['UMAP_Y'] = umap_result[:, 1]


In [7]:
pca = pd.read_csv(f"{init.root_data_results_dir}pca_loadings.csv")
# master_loadings = pd.read_csv(f"{init.root_data_results_dir}tableau_ready.csv")
df_trends_2011 = pd.read_csv(f"{init.root_data_results_dir}2011_analysis.csv")
df_2011_vs_2021 = pd.read_csv(f"{init.root_data_results_dir}2011_2021_comparison.csv")

In [8]:
master_loadings.shape

(636, 122)

In [9]:
print(master_loadings.columns)

Index(['area', 'district_code', 'total_population', 'white_total',
       'white_british', 'white_irish', 'white_gypsy_traveller', 'white_other',
       'mixed_total', 'mixed_white_black_caribbean',
       ...
       'public_admin_and_defence_prop', 'education_prop',
       'health_and_social_work_prop', 'other_services_prop', 'PCA_1', 'PCA_2',
       'PCA_3', 'PCA_4', 'TSNE_X', 'TSNE_Y'],
      dtype='str', length=122)


In [10]:
print(pca.head())

              Unnamed: 0  PC1_Weight  PC2_Weight  PC3_Weight  PC4_Weight
0          white_british   -0.163555   -0.145949   -0.200762   -0.097419
1            white_irish    0.134038    0.193342    0.023177   -0.041405
2  white_gypsy_traveller    0.007112   -0.018605   -0.076972    0.261278
3            white_other    0.171708    0.129888    0.078792   -0.035809
4      asian_bangladeshi    0.059656    0.059498    0.108997   -0.058075


In [11]:
master_loadings.columns, master_loadings.shape

(Index(['area', 'district_code', 'total_population', 'white_total',
        'white_british', 'white_irish', 'white_gypsy_traveller', 'white_other',
        'mixed_total', 'mixed_white_black_caribbean',
        ...
        'public_admin_and_defence_prop', 'education_prop',
        'health_and_social_work_prop', 'other_services_prop', 'PCA_1', 'PCA_2',
        'PCA_3', 'PCA_4', 'TSNE_X', 'TSNE_Y'],
       dtype='str', length=122),
 (636, 122))

In [12]:
pca_sort = pca.sort_values(by="PC1_Weight")

In [13]:
pca_sort[["Unnamed: 0", "PC1_Weight"]].head()

,Unnamed: 0,PC1_Weight
36,water_and_waste_management,-0.191445
34,manufacturing,-0.189952
37,construction,-0.187358
38,wholesale_and_retail,-0.184566
48,health_and_social_work,-0.183348


In [14]:
pca_sort[["Unnamed: 0", "PC1_Weight"]].tail()

,Unnamed: 0,PC1_Weight
13,mixed_white_black_african,0.172792
17,other_remaining,0.174384
24,professional_occupations,0.177411
15,mixed_other,0.193492
12,mixed_white_asian,0.195585


In [15]:
pca_sort2 = pca.sort_values(by="PC2_Weight")

In [16]:
pca_sort2[["Unnamed: 0", "PC2_Weight"]].head()

,Unnamed: 0,PC2_Weight
20,fair_health,-0.153339
0,white_british,-0.145949
28,caring_and_leisure,-0.143131
27,skilled_trades_occupations,-0.120109
30,process_plant_and_machine_operatives,-0.118734


In [17]:
pca_sort2[["Unnamed: 0", "PC2_Weight"]].tail()

,Unnamed: 0,PC2_Weight
42,financial_and_insurance,0.213505
49,other_services,0.218572
43,real_estate_activities,0.218973
41,information_and_communication,0.233837
44,professional_and_scientific,0.235529


In [18]:
pca_sort3 = pca.sort_values(by="PC3_Weight")

In [19]:
pca_sort3[["Unnamed: 0", "PC3_Weight"]].head()

,Unnamed: 0,PC3_Weight
23,managers_directors_and_senior_officials,-0.299008
0,white_british,-0.200762
25,associate_prof_and_tech,-0.189972
18,very_good_health,-0.175444
24,professional_occupations,-0.148264


In [20]:
pca_sort3[["Unnamed: 0", "PC3_Weight"]].tail()

,Unnamed: 0,PC3_Weight
30,process_plant_and_machine_operatives,0.241217
21,bad_health,0.241957
22,very_bad_health,0.262463
29,sales_and_customer_service,0.266534
31,elementary_occupations,0.315725


In [22]:
df_trends_2011

,area,district_code,total_population,white_total,white_british,white_irish,white_gypsy_traveller,white_other,mixed_total,mixed_white_black_caribbean,...,public_admin_and_defence_prop,education_prop,health_and_social_work_prop,other_services_prop,PCA_1,PCA_2,PCA_3,PCA_4,TSNE_X,TSNE_Y
0,Adur,E07000223,61182,58663,56843,426,121,1273,886,206,...,0.047452,0.091532,0.142083,0.049700,-2.184982,-0.143825,0.615361,-0.402991,-13.302950,-1.570238
1,Allerdale,E07000026,96422,95385,94085,241,14,1045,426,134,...,0.060639,0.083275,0.118869,0.043949,-5.150425,0.170773,-1.511428,1.507831,-6.687267,10.091799
2,Amber Valley,E07000032,122309,120023,117988,428,44,1563,980,438,...,0.052003,0.094138,0.119213,0.043633,-3.632313,0.406120,0.203937,-0.130103,-1.850601,9.911616
3,Arun,E07000224,149518,145118,137024,889,162,7043,1502,426,...,0.053645,0.083092,0.143988,0.055024,-2.689894,-0.284468,-0.099545,1.509442,-13.577929,1.928089
4,Ashfield,E07000170,119497,116733,114744,348,43,1598,1059,556,...,0.051811,0.076352,0.135066,0.037552,-5.200480,-1.990276,0.299757,-0.637144,-1.132346,14.393061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,Wychavon,E07000238,116944,114457,109834,539,462,3622,962,339,...,0.056137,0.089878,0.107744,0.046029,-1.486649,2.238317,1.502472,1.435741,-7.123997,-7.445074
344,Wycombe,E07000007,171644,139477,130313,1553,103,7508,4849,2259,...,0.050295,0.097707,0.095171,0.052762,3.155423,1.425657,1.958686,-0.234413,4.847900,-9.274721
345,Wyre,E07000128,107749,105852,104251,490,96,1015,664,234,...,0.108159,0.098892,0.133758,0.046295,-3.614902,0.542712,-1.391846,-0.817191,-9.804702,12.332017
346,Wyre Forest,E07000239,97975,95273,92822,459,375,1617,998,461,...,0.047627,0.089665,0.124172,0.049548,-3.132988,0.129573,0.733071,0.770305,-8.368806,-7.058453


In [23]:
df_2011_vs_2021

,area,district_code,total_population,white_total,white_british,white_irish,white_gypsy_traveller,white_other,mixed_total,mixed_white_black_caribbean,...,financial_and_insurance_prop,real_estate_activities_prop,professional_and_scientific_prop,admin_and_support_services_prop,public_admin_and_defence_prop,education_prop,health_and_social_work_prop,other_services_prop,UMAP_X,UMAP_Y
0,Adur,E07000223,61182,58663,56843,426,121,1273,886,206,...,0.054946,0.015704,0.053209,0.045374,0.047452,0.091532,0.142083,0.049700,-1.793981,0.408857
1,Amber Valley,E07000032,122309,120023,117988,428,44,1563,980,438,...,0.016184,0.012058,0.048197,0.038295,0.052003,0.094138,0.119213,0.043633,-1.172631,0.036301
2,Arun,E07000224,149518,145118,137024,889,162,7043,1502,426,...,0.032235,0.015598,0.049508,0.052029,0.053645,0.083092,0.143988,0.055024,-2.276594,-0.548545
3,Ashfield,E07000170,119497,116733,114744,348,43,1598,1059,556,...,0.016300,0.010066,0.036504,0.041637,0.051811,0.076352,0.135066,0.037552,-0.514041,-0.015455
4,Ashford,E07000105,117956,110520,105498,753,531,3738,1682,544,...,0.033625,0.011295,0.061151,0.054826,0.050238,0.086922,0.126073,0.049195,-3.462883,0.279825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
631,Wrexham,W06000006,135122,129677,122547,427,119,6584,1558,382,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.584153,1.809286
632,Wychavon,E07000238,132495,128325,120048,624,637,7016,1600,481,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,16.856337,3.498149
633,Wyre,E07000128,111946,109139,107171,487,71,1410,1135,302,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.850309,1.425673
634,Wyre Forest,E07000239,101609,97423,93961,475,451,2536,1470,606,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,16.358934,1.745010
